# 🚇 Checkpoint 2 — Dynamic Programming: Rotas de Metrô

## [1] Identificação do Grupo

| Nome Completo | RM

Christian Schunck de Almeida| RM-563850 
Guilherme Vilela Perez | RM-564422 
Gustavo Panham Dourado | RM-563904
Paulo Cesar de Govea Junior | RM-566034 
Thomas Jeserfon Santana Wang | RM-565104 

**Disciplina:** FIAP — Dynamic Programming  
**Checkpoint:** Checkpoint 2 — Rotas de Metrô com Recursão e Memoização  

## [2] Modelagem dos Grafos

### Teoria
Cada rede de metrô é representada como um **grafo ponderado não-dirigido**:
- **Nós** = estações
- **Arestas** = conexões físicas entre estações adjacentes
- **Pesos** = tempo estimado de viagem entre estações (em minutos)

**Justificativa para grafo não-dirigido:** os trens percorrem os mesmos trilhos nos dois sentidos, com o mesmo tempo de trajeto. Em casos reais de linhas expressas ou sentido único, utilizaríamos dígrafos.

**Complexidade de espaço:** O(V + E), onde V = número de vértices (estações) e E = número de arestas (conexões).


In [1]:
# ─────────────────────────────────────────────────────────────────────────────
# GRAFOS DAS TRÊS CIDADES
# Formato: { estação: [(vizinho, peso_minutos), ...] }
# Grafos não-dirigidos → cada aresta aparece nos dois sentidos
# ─────────────────────────────────────────────────────────────────────────────

# ══════════════════════════════════════════════════════════════════════════════
# BEIJING — Linhas 1, 2, 4 e 10
# Origem: Sihui East | Destino: Xizhimen
# ══════════════════════════════════════════════════════════════════════════════
grafo_beijing = {
    # Linha 1 (Leste→Oeste)
    "Sihui East":       [("Sihui", 2)],
    "Sihui":            [("Sihui East", 2), ("Guomao", 3)],
    "Guomao":           [("Sihui", 3), ("Dawanglu", 3)],
    "Dawanglu":         [("Guomao", 3), ("Yong'anli", 3)],
    "Yong'anli":        [("Dawanglu", 3), ("Jianguomen", 3)],
    "Jianguomen":       [("Yong'anli", 3), ("Beijing Station", 4), ("Jianguomen L2", 2)],
    "Beijing Station":  [("Jianguomen", 4), ("Wangfujing", 4)],
    "Wangfujing":       [("Beijing Station", 4), ("Tian'anmen East", 3)],
    "Tian'anmen East":  [("Wangfujing", 3), ("Tian'anmen West", 3)],
    "Tian'anmen West":  [("Tian'anmen East", 3), ("Xidan", 3)],
    "Xidan":            [("Tian'anmen West", 3), ("Fuxingmen", 3), ("Xidan L4", 2)],
    "Fuxingmen":        [("Xidan", 3), ("Muxidi", 3), ("Fuxingmen L2", 2)],
    "Muxidi":           [("Fuxingmen", 3), ("Junshi Bowuguan", 3)],
    "Junshi Bowuguan":  [("Muxidi", 3), ("Gongzhufen", 3)],
    "Gongzhufen":       [("Junshi Bowuguan", 3), ("Wanshoulu", 3), ("Gongzhufen L10", 2)],
    "Wanshoulu":        [("Gongzhufen", 3)],
    # Linha 2 (Anel) — nós de integração e conexão com L1
    "Jianguomen L2":    [("Jianguomen", 2), ("Chaoyangmen", 3), ("Beijing Station L2", 3)],
    "Beijing Station L2":[("Jianguomen L2", 3), ("Chongwenmen", 3)],
    "Chongwenmen":      [("Beijing Station L2", 3), ("Qianmen", 3)],
    "Qianmen":          [("Chongwenmen", 3), ("Hepingmen", 3)],
    "Hepingmen":        [("Qianmen", 3), ("Xuanwumen", 3)],
    "Xuanwumen":        [("Hepingmen", 3), ("Changchunjie", 3)],
    "Changchunjie":     [("Xuanwumen", 3), ("Fuxingmen L2", 3)],
    "Fuxingmen L2":     [("Changchunjie", 3), ("Fuxingmen", 2), ("Chegongzhuang", 3)],
    "Chegongzhuang":    [("Fuxingmen L2", 3), ("Xizhimen", 4)],
    "Chaoyangmen":      [("Jianguomen L2", 3), ("Dongsishitiao", 3)],
    "Dongsishitiao":    [("Chaoyangmen", 3), ("Dongzhimen", 3)],
    "Dongzhimen":       [("Dongsishitiao", 3), ("Guloudajie", 4)],
    "Guloudajie":       [("Dongzhimen", 4), ("Jishuitan", 3)],
    "Jishuitan":        [("Guloudajie", 3), ("Xizhimen", 3)],
    "Xizhimen":         [("Jishuitan", 3), ("Chegongzhuang", 4), ("Xizhimen L4", 2)],
    # Linha 4
    "Xidan L4":         [("Xidan", 2), ("Xizhimen L4", 6)],
    "Xizhimen L4":      [("Xidan L4", 6), ("Xizhimen", 2)],
    # Linha 10
    "Gongzhufen L10":   [("Gongzhufen", 2), ("Sanyuanqiao", 6)],
    "Sanyuanqiao":      [("Gongzhufen L10", 6)],
}

# ══════════════════════════════════════════════════════════════════════════════
# SAN FRANCISCO — Rede BART
# Origem: Dublin/Pleasanton | Destino: Daly City
# ══════════════════════════════════════════════════════════════════════════════
grafo_sf = {
    # Linha Azul (Blue) Dublin→Daly City via Oakland
    "Dublin/Pleasanton":  [("West Dublin", 4)],
    "West Dublin":        [("Dublin/Pleasanton", 4), ("Castro Valley", 6)],
    "Castro Valley":      [("West Dublin", 6), ("Bay Fair", 5)],
    "Bay Fair":           [("Castro Valley", 5), ("San Leandro", 4), ("South Hayward", 4)],
    "San Leandro":        [("Bay Fair", 4), ("Fruitvale", 5)],
    "Fruitvale":          [("San Leandro", 5), ("Coliseum", 4)],
    "Coliseum":           [("Fruitvale", 4), ("Lake Merritt", 5)],
    "Lake Merritt":       [("Coliseum", 5), ("West Oakland", 8)],
    "West Oakland":       [("Lake Merritt", 8), ("Embarcadero", 7)],
    "Embarcadero":        [("West Oakland", 7), ("Montgomery St", 2), ("Embarcadero Y", 3)],
    "Montgomery St":      [("Embarcadero", 2), ("Powell St", 2)],
    "Powell St":          [("Montgomery St", 2), ("Civic Center", 2)],
    "Civic Center":       [("Powell St", 2), ("16th St Mission", 3)],
    "16th St Mission":    [("Civic Center", 3), ("24th St Mission", 2)],
    "24th St Mission":    [("16th St Mission", 2), ("Glen Park", 3)],
    "Glen Park":          [("24th St Mission", 3), ("Balboa Park", 3)],
    "Balboa Park":        [("Glen Park", 3), ("Daly City", 4), ("Balboa Y", 2)],
    "Daly City":          [("Balboa Park", 4)],
    # Linha Amarela (Yellow) — Pittsburg via Concord
    "Embarcadero Y":      [("Embarcadero", 3), ("12th St Oakland", 5)],
    "12th St Oakland":    [("Embarcadero Y", 5), ("19th St Oakland", 2)],
    "19th St Oakland":    [("12th St Oakland", 2), ("MacArthur", 3)],
    "MacArthur":          [("19th St Oakland", 3), ("Rockridge", 4), ("MacArthur G", 3)],
    "Rockridge":          [("MacArthur", 4), ("Orinda", 7)],
    "Orinda":             [("Rockridge", 7), ("Lafayette", 4)],
    "Lafayette":          [("Orinda", 4), ("Walnut Creek", 5)],
    "Walnut Creek":       [("Lafayette", 5)],
    # Linha Verde — Fremont
    "MacArthur G":        [("MacArthur", 3), ("South Hayward", 8)],
    "South Hayward":      [("Bay Fair", 4), ("MacArthur G", 8), ("Union City", 4)],
    "Union City":         [("South Hayward", 4), ("Fremont", 5)],
    "Fremont":            [("Union City", 5)],
    # Bifurcação Balboa
    "Balboa Y":           [("Balboa Park", 2), ("Colma", 4)],
    "Colma":              [("Balboa Y", 4)],
}

# ══════════════════════════════════════════════════════════════════════════════
# SÃO PAULO — Metrô + CPTM
# Origem: Tucuruvi (L1-Azul) | Destino: Capão Redondo (L5-Lilás)
# Integração obrigatória entre linhas
# ══════════════════════════════════════════════════════════════════════════════
grafo_sp = {
    # Linha 1 — Azul (Tucuruvi → Jabaquara)
    "Tucuruvi":           [("Parada Inglesa", 2)],
    "Parada Inglesa":     [("Tucuruvi", 2), ("Jardim São Paulo", 2)],
    "Jardim São Paulo":   [("Parada Inglesa", 2), ("Carandiru", 2)],
    "Carandiru":          [("Jardim São Paulo", 2), ("Santana", 2)],
    "Santana":            [("Carandiru", 2), ("Portuguesa-Tietê", 2)],
    "Portuguesa-Tietê":   [("Santana", 2), ("Armênia", 2)],
    "Armênia":            [("Portuguesa-Tietê", 2), ("Tiradentes", 2)],
    "Tiradentes":         [("Armênia", 2), ("Luz", 3)],
    # Luz — integração L1+L3+CPTM
    "Luz":                [("Tiradentes", 3), ("São Bento", 2), ("Luz L3", 0), ("Luz CPTM", 0)],
    "São Bento":          [("Luz", 2), ("Sé", 2)],
    # Sé — integração L1+L3
    "Sé":                 [("São Bento", 2), ("Liberdade", 2), ("Sé L3", 0)],
    "Liberdade":          [("Sé", 2), ("Paraíso", 3)],
    # Paraíso — integração L1+L2
    "Paraíso":            [("Liberdade", 3), ("Ana Rosa", 2), ("Paraíso L2", 0)],
    "Ana Rosa":           [("Paraíso", 2), ("Praça da Árvore", 3)],
    "Praça da Árvore":    [("Ana Rosa", 3), ("Saúde", 2)],
    "Saúde":              [("Praça da Árvore", 2), ("Jabaquara", 3)],
    "Jabaquara":          [("Saúde", 3)],
    # Linha 2 — Verde (Vila Madalena → Vila Prudente)
    "Paraíso L2":         [("Paraíso", 0), ("Brigadeiro", 2), ("Consolação", 4)],
    "Brigadeiro":         [("Paraíso L2", 2), ("Trianon-Masp", 2)],
    "Trianon-Masp":       [("Brigadeiro", 2), ("Consolação", 2)],
    "Consolação":         [("Trianon-Masp", 2), ("Paraíso L2", 4), ("Paulista L4", 2)],
    # Paulista — integração L2+L4
    "Paulista L4":        [("Consolação", 2), ("Higienópolis-Mackenzie", 3)],
    "Higienópolis-Mackenzie": [("Paulista L4", 3)],
    # Linha 3 — Vermelha (Palmeiras-Barra Funda → Corinthians-Itaquera)
    "Luz L3":             [("Luz", 0), ("Sé L3", 3), ("Marechal Deodoro", 4)],
    "Sé L3":              [("Sé", 0), ("Luz L3", 3), ("Santa Cruz L3", 6)],
    "Marechal Deodoro":   [("Luz L3", 4), ("Palmeiras-Barra Funda", 4)],
    "Palmeiras-Barra Funda": [("Marechal Deodoro", 4)],
    "Santa Cruz L3":      [("Sé L3", 6), ("Clínicas", 4)],
    "Clínicas":           [("Santa Cruz L3", 4), ("Vila Mariana", 5)],
    "Vila Mariana":       [("Clínicas", 5)],
    # Linha 4 — Amarela (Butantã → Luz) — acesso a L5 via baldeação
    "Butantã L4":         [("Paulista L4", 5), ("Pinheiros L4", 4)],
    "Pinheiros L4":       [("Butantã L4", 4), ("Faria Lima", 3)],
    "Faria Lima":         [("Pinheiros L4", 3), ("Fradique Coutinho", 2)],
    "Fradique Coutinho":  [("Faria Lima", 2), ("Oscar Freire", 2)],
    "Oscar Freire":       [("Fradique Coutinho", 2)],
    # Linha 5 — Lilás (Capão Redondo → Chácara Klabin)
    # integração com L2 em Santo Amaro→Chácara Klabin (simplificado)
    "Capão Redondo":      [("Campo Limpo", 3)],
    "Campo Limpo":        [("Capão Redondo", 3), ("Vila das Belezas", 2)],
    "Vila das Belezas":   [("Campo Limpo", 2), ("Giovanni Gronchi", 2)],
    "Giovanni Gronchi":   [("Vila das Belezas", 2), ("Santo Amaro", 3)],
    "Santo Amaro":        [("Giovanni Gronchi", 3), ("Adolfo Pinheiro", 2), ("Santo Amaro CPTM", 0)],
    "Adolfo Pinheiro":    [("Santo Amaro", 2), ("Long. Iguatemi", 3)],
    "Long. Iguatemi":     [("Adolfo Pinheiro", 3), ("Eucaliptos", 2)],
    "Eucaliptos":         [("Long. Iguatemi", 2), ("Moema", 3)],
    "Moema":              [("Eucaliptos", 3), ("AACD-Servidor", 2)],
    "AACD-Servidor":      [("Moema", 2), ("Hospital São Paulo", 2)],
    "Hospital São Paulo":  [("AACD-Servidor", 2), ("Santa Cruz L5", 2)],
    # Santa Cruz — integração L3+L5
    "Santa Cruz L5":      [("Hospital São Paulo", 2), ("Sé L3", 5)],
    # CPTM (simplificado para conectar Santo Amaro → Luz)
    "Luz CPTM":           [("Luz", 0), ("Santo Amaro CPTM", 15)],
    "Santo Amaro CPTM":   [("Santo Amaro", 0), ("Luz CPTM", 15)],
}

print(f"Nós Beijing: {len(grafo_beijing)}")
print(f"Nós San Francisco: {len(grafo_sf)}")
print(f"Nós São Paulo: {len(grafo_sp)}")


Nós Beijing: 35
Nós San Francisco: 32
Nós São Paulo: 49


## [3] Fatores de Horário — Penalidades e Bônus

| Faixa Horária | Fator | Justificativa |
|---|---|---|
| 05h–07h | × 0,6 | Metrô vazio, embarque rápido |
| 07h–09h | × 1,5 | Pico da manhã, volume alto |
| 09h–17h | × 1,0 | Fluxo regular |
| 17h–20h | × 2,0 | Pico da tarde, lotação máxima |
| 20h–05h | × 0,8 | Noturno, fluxo reduzido |


In [2]:
def fator_horario(hora: int) -> float:
    """Retorna o multiplicador de custo conforme a faixa horária."""
    if 5 <= hora < 7:
        return 0.6   # Bônus madrugada/manhã cedo
    elif 7 <= hora < 9:
        return 1.5   # Pico da manhã
    elif 9 <= hora < 17:
        return 1.0   # Horário normal
    elif 17 <= hora < 20:
        return 2.0   # Pico da tarde (penalidade máxima)
    else:
        return 0.8   # Noturno

# Teste
for h in [6, 8, 12, 18, 22]:
    print(f"  Hora {h:02d}h → fator {fator_horario(h)}")


  Hora 06h → fator 0.6
  Hora 08h → fator 1.5
  Hora 12h → fator 1.0
  Hora 18h → fator 2.0
  Hora 22h → fator 0.8


## [4] Implementação: Recursão + Memoização

### Algoritmo — Caminho Mais Curto
Utilizamos **recursão com memoização** (`functools.lru_cache`).  
O estado é definido pelo par `(origem, destino, visitados)` — onde `visitados` é um `frozenset` hashável.

**Complexidade:**
- **Sem memoização:** O(V!) no pior caso (todas as permutações de caminhos simples)
- **Com memoização:** O(V · 2^V) — cada subproblema `(nó, conjunto_visitados)` é resolvido uma única vez

### Algoritmo — Caminho Mais Longo Simples
Usa **backtracking** puro (sem memoização, pois o problema de caminho mais longo simples é NP-difícil e os subproblemas não têm subestrutura ótima independente do conjunto de visitados).


In [3]:
import functools

def resolver_cidade(grafo, origem, destino, hora):
    """
    Resolve o caminho mais curto (recursão+memoização) e
    o caminho mais longo simples (backtracking) para uma cidade.
    """
    fator = fator_horario(hora)

    # ── Caminho mais curto com memoização ────────────────────────────────────
    @functools.lru_cache(maxsize=None)
    def menor_custo(atual, visitados):
        if atual == destino:
            return (0, (destino,))
        melhor_custo = float('inf')
        melhor_caminho = None
        vizinhos = grafo.get(atual, [])
        for viz, peso in vizinhos:
            if viz not in visitados:
                custo_sub, cam_sub = menor_custo(viz, visitados | frozenset([atual]))
                custo_total = fator * peso + custo_sub
                if custo_total < melhor_custo:
                    melhor_custo = custo_total
                    melhor_caminho = (atual,) + cam_sub
        return (melhor_custo, melhor_caminho if melhor_caminho else ())

    custo_min, caminho_min = menor_custo(origem, frozenset())
    menor_custo.cache_clear()

    # ── Caminho mais longo simples com backtracking ───────────────────────────
    melhor_longo = [float('-inf'), []]

    def maior_custo(atual, visitados, custo_acum, caminho):
        if atual == destino:
            if custo_acum > melhor_longo[0]:
                melhor_longo[0] = custo_acum
                melhor_longo[1] = caminho[:]
            return
        for viz, peso in grafo.get(atual, []):
            if viz not in visitados:
                visitados.add(viz)
                caminho.append(viz)
                maior_custo(viz, visitados, custo_acum + fator * peso, caminho)
                caminho.pop()
                visitados.remove(viz)

    maior_custo(origem, {origem}, 0, [origem])

    return {
        "custo_min": round(custo_min, 2),
        "caminho_min": list(caminho_min),
        "custo_max": round(melhor_longo[0], 2),
        "caminho_max": melhor_longo[1],
    }

# Teste rápido com SP
res = resolver_cidade(grafo_sp, "Tucuruvi", "Capão Redondo", hora=12)
print("✅ Caminho mais curto — custo:", res['custo_min'])
print("   Estações:", len(res['caminho_min']))
print("✅ Caminho mais longo — custo:", res['custo_max'])
print("   Estações:", len(res['caminho_max']))


✅ Caminho mais curto — custo: 42.0
   Estações: 16
✅ Caminho mais longo — custo: 42.0
   Estações: 16


## [5] Análise de Desempenho — Tempo e Memória

Comparamos a execução **com e sem memoização** para cada cidade, usando `time.perf_counter` e `tracemalloc`.


In [4]:
import time, tracemalloc

def medir_desempenho(grafo, origem, destino, hora, com_memo=True):
    """Mede tempo e memória do algoritmo de caminho mais curto."""
    fator = fator_horario(hora)

    if com_memo:
        @functools.lru_cache(maxsize=None)
        def menor_custo(atual, visitados):
            if atual == destino:
                return 0
            melhor = float('inf')
            for viz, peso in grafo.get(atual, []):
                if viz not in visitados:
                    c = fator * peso + menor_custo(viz, visitados | frozenset([atual]))
                    melhor = min(melhor, c)
            return melhor
    else:
        def menor_custo(atual, visitados):
            if atual == destino:
                return 0
            melhor = float('inf')
            for viz, peso in grafo.get(atual, []):
                if viz not in visitados:
                    c = fator * peso + menor_custo(viz, visitados | frozenset([atual]))
                    melhor = min(melhor, c)
            return melhor

    tracemalloc.start()
    t0 = time.perf_counter()
    resultado = menor_custo(origem, frozenset())
    t1 = time.perf_counter()
    mem_atual, mem_pico = tracemalloc.get_traced_memory()
    tracemalloc.stop()

    return {
        "resultado": round(resultado, 2),
        "tempo_s": round(t1 - t0, 6),
        "mem_pico_kb": round(mem_pico / 1024, 2),
    }

# ── Executar para as 3 cidades ───────────────────────────────────────────────
cidades = [
    ("Beijing",       grafo_beijing, "Sihui East",       "Xizhimen"),
    ("San Francisco", grafo_sf,      "Dublin/Pleasanton","Daly City"),
    ("São Paulo",     grafo_sp,      "Tucuruvi",         "Capão Redondo"),
]
HORA_TESTE = 18  # pico da tarde

resultados_desempenho = []
for nome, grafo, orig, dest in cidades:
    com    = medir_desempenho(grafo, orig, dest, HORA_TESTE, com_memo=True)
    sem    = medir_desempenho(grafo, orig, dest, HORA_TESTE, com_memo=False)
    resultados_desempenho.append((nome, com, sem))
    print(f"{'─'*60}")
    print(f"🏙  {nome}")
    print(f"  COM memo  → custo: {com['resultado']} | tempo: {com['tempo_s']:.6f}s | mem: {com['mem_pico_kb']:.1f} KB")
    print(f"  SEM memo  → custo: {sem['resultado']} | tempo: {sem['tempo_s']:.6f}s | mem: {sem['mem_pico_kb']:.1f} KB")
print(f"{'─'*60}")


────────────────────────────────────────────────────────────
🏙  Beijing
  COM memo  → custo: 70.0 | tempo: 0.000609s | mem: 55.7 KB
  SEM memo  → custo: 70.0 | tempo: 0.000532s | mem: 21.3 KB
────────────────────────────────────────────────────────────
🏙  San Francisco
  COM memo  → custo: 128.0 | tempo: 0.000594s | mem: 44.4 KB
  SEM memo  → custo: 128.0 | tempo: 0.000468s | mem: 14.0 KB
────────────────────────────────────────────────────────────
🏙  São Paulo
  COM memo  → custo: 84.0 | tempo: 0.000910s | mem: 95.7 KB
  SEM memo  → custo: 84.0 | tempo: 0.000828s | mem: 25.5 KB
────────────────────────────────────────────────────────────


In [5]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

nomes   = [r[0] for r in resultados_desempenho]
t_com   = [r[1]['tempo_s']*1000  for r in resultados_desempenho]
t_sem   = [r[2]['tempo_s']*1000  for r in resultados_desempenho]
m_com   = [r[1]['mem_pico_kb']   for r in resultados_desempenho]
m_sem   = [r[2]['mem_pico_kb']   for r in resultados_desempenho]

x = np.arange(len(nomes))
w = 0.3

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Tempo
ax1.bar(x - w/2, t_com, w, label='Com memoização', color='#2196F3', alpha=0.85)
ax1.bar(x + w/2, t_sem, w, label='Sem memoização', color='#F44336', alpha=0.85)
ax1.set_title('Tempo de Execução (ms)', fontsize=13, fontweight='bold')
ax1.set_xticks(x); ax1.set_xticklabels(nomes)
ax1.set_ylabel('Tempo (ms)'); ax1.legend()
ax1.grid(axis='y', alpha=0.3)

# Memória
ax2.bar(x - w/2, m_com, w, label='Com memoização', color='#4CAF50', alpha=0.85)
ax2.bar(x + w/2, m_sem, w, label='Sem memoização', color='#FF9800', alpha=0.85)
ax2.set_title('Memória de Pico (KB)', fontsize=13, fontweight='bold')
ax2.set_xticks(x); ax2.set_xticklabels(nomes)
ax2.set_ylabel('Memória (KB)'); ax2.legend()
ax2.grid(axis='y', alpha=0.3)

plt.suptitle('Análise de Desempenho: Com vs Sem Memoização', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('/mnt/user-data/outputs/desempenho.png', dpi=150, bbox_inches='tight')
plt.close()
print("Gráfico salvo.")


Gráfico salvo.


## [6] Resultados por Cidade

Abaixo executamos os dois algoritmos (menor e maior custo) para cada cidade nas diferentes faixas horárias.


In [6]:
# ── Resultados completos para cada cidade e faixa horária ────────────────────
horas_teste = [6, 8, 12, 18]
nomes_faixas = {6:"Madrugada(×0.6)", 8:"Pico Manhã(×1.5)",
                12:"Normal(×1.0)", 18:"Pico Tarde(×2.0)"}

for nome_cidade, grafo, orig, dest in cidades:
    print(f"\n{'═'*65}")
    print(f"  🏙  {nome_cidade} | {orig} → {dest}")
    print(f"{'═'*65}")
    for h in horas_teste:
        res = resolver_cidade(grafo, orig, dest, hora=h)
        print(f"  ⏰ {nomes_faixas[h]}")
        print(f"     Mais curto : custo={res['custo_min']:7.1f} | {len(res['caminho_min'])} estações")
        print(f"       Rota: {' → '.join(res['caminho_min'][:5])}{'...' if len(res['caminho_min'])>5 else ''}")
        print(f"     Mais longo : custo={res['custo_max']:7.1f} | {len(res['caminho_max'])} estações")
        print()



═════════════════════════════════════════════════════════════════
  🏙  Beijing | Sihui East → Xizhimen
═════════════════════════════════════════════════════════════════
  ⏰ Madrugada(×0.6)
     Mais curto : custo=   21.0 | 13 estações
       Rota: Sihui East → Sihui → Guomao → Dawanglu → Yong'anli...
     Mais longo : custo=   45.6 | 26 estações

  ⏰ Pico Manhã(×1.5)
     Mais curto : custo=   52.5 | 13 estações
       Rota: Sihui East → Sihui → Guomao → Dawanglu → Yong'anli...
     Mais longo : custo=  114.0 | 26 estações

  ⏰ Normal(×1.0)
     Mais curto : custo=   35.0 | 13 estações
       Rota: Sihui East → Sihui → Guomao → Dawanglu → Yong'anli...
     Mais longo : custo=   76.0 | 26 estações

  ⏰ Pico Tarde(×2.0)
     Mais curto : custo=   70.0 | 13 estações
       Rota: Sihui East → Sihui → Guomao → Dawanglu → Yong'anli...
     Mais longo : custo=  152.0 | 26 estações


═════════════════════════════════════════════════════════════════
  🏙  San Francisco | Dublin/Pleasanton → Dal

## [7] Visualização com Folium

Exibimos o grafo das estações sobre o mapa real. As arestas são desenhadas em cinza; o **caminho mais curto** é destacado em **azul** e o **mais longo** em **vermelho**.


In [7]:
import folium

# Coordenadas das estações (lat, lon)
coords_sp = {
    "Tucuruvi":           (-23.4753, -46.6073),
    "Parada Inglesa":     (-23.4851, -46.6133),
    "Jardim São Paulo":   (-23.4948, -46.6183),
    "Carandiru":          (-23.5039, -46.6262),
    "Santana":            (-23.5062, -46.6281),
    "Portuguesa-Tietê":  (-23.5121, -46.6351),
    "Armênia":            (-23.5193, -46.6395),
    "Tiradentes":         (-23.5283, -46.6354),
    "Luz":                (-23.5364, -46.6345),
    "São Bento":          (-23.5394, -46.6341),
    "Sé":                 (-23.5504, -46.6334),
    "Liberdade":          (-23.5587, -46.6328),
    "Paraíso":            (-23.5742, -46.6481),
    "Ana Rosa":           (-23.5810, -46.6432),
    "Praça da Árvore":   (-23.5884, -46.6341),
    "Saúde":              (-23.5957, -46.6265),
    "Jabaquara":          (-23.6265, -46.6243),
    # L5
    "Capão Redondo":      (-23.6679, -46.7775),
    "Campo Limpo":        (-23.6589, -46.7630),
    "Vila das Belezas":   (-23.6518, -46.7491),
    "Giovanni Gronchi":   (-23.6386, -46.7301),
    "Santo Amaro":        (-23.6515, -46.7083),
    "Adolfo Pinheiro":    (-23.6408, -46.6978),
    "Long. Iguatemi":    (-23.6335, -46.6851),
    "Eucaliptos":         (-23.6218, -46.6720),
    "Moema":              (-23.6076, -46.6642),
    "AACD-Servidor":     (-23.5972, -46.6581),
    "Hospital São Paulo": (-23.5904, -46.6536),
    "Santa Cruz L5":      (-23.5836, -46.6497),
    # integrações / CPTM
    "Luz CPTM":           (-23.5370, -46.6360),
    "Santo Amaro CPTM":   (-23.6520, -46.7090),
    "Paraíso L2":         (-23.5748, -46.6488),
    "Consolação":         (-23.5549, -46.6618),
    "Paulista L4":        (-23.5614, -46.6558),
    "Brigadeiro":         (-23.5681, -46.6531),
    "Trianon-Masp":       (-23.5628, -46.6572),
    "Higienópolis-Mackenzie": (-23.5486, -46.6612),
    "Luz L3":             (-23.5368, -46.6342),
    "Sé L3":              (-23.5505, -46.6335),
    "Marechal Deodoro":   (-23.5361, -46.6451),
    "Palmeiras-Barra Funda": (-23.5257, -46.6614),
    "Santa Cruz L3":      (-23.5838, -46.6432),
    "Clínicas":           (-23.5573, -46.6724),
    "Vila Mariana":       (-23.5887, -46.6378),
    "Butantã L4":         (-23.5714, -46.7168),
    "Pinheiros L4":       (-23.5663, -46.6981),
    "Faria Lima":         (-23.5702, -46.6882),
    "Fradique Coutinho":  (-23.5641, -46.6832),
    "Oscar Freire":       (-23.5608, -46.6801),
    "Xidan L4":           (-23.5400, -46.6800),
    "Xizhimen L4":        (-23.5200, -46.6600),
}

def criar_mapa_sp(hora_vis=12):
    res_vis = resolver_cidade(grafo_sp, "Tucuruvi", "Capão Redondo", hora=hora_vis)
    cam_min = res_vis['caminho_min']
    cam_max = res_vis['caminho_max']

    m = folium.Map(location=[-23.57, -46.67], zoom_start=12, tiles='CartoDB positron')

    # Todas as arestas (cinza)
    visitadas_arestas = set()
    for estacao, vizinhos in grafo_sp.items():
        for viz, _ in vizinhos:
            par = tuple(sorted([estacao, viz]))
            if par not in visitadas_arestas:
                if estacao in coords_sp and viz in coords_sp:
                    folium.PolyLine(
                        [coords_sp[estacao], coords_sp[viz]],
                        color='gray', weight=2, opacity=0.4
                    ).add_to(m)
                visitadas_arestas.add(par)

    # Caminho mais longo (vermelho tracejado)
    coords_max = [coords_sp[e] for e in cam_max if e in coords_sp]
    if len(coords_max) >= 2:
        folium.PolyLine(coords_max, color='red', weight=3,
                        opacity=0.7, dash_array='8 4',
                        tooltip='Caminho mais longo').add_to(m)

    # Caminho mais curto (azul sólido)
    coords_min = [coords_sp[e] for e in cam_min if e in coords_sp]
    if len(coords_min) >= 2:
        folium.PolyLine(coords_min, color='#1565C0', weight=5,
                        opacity=0.9, tooltip='Caminho mais curto').add_to(m)

    # Marcadores
    for est, (lat, lon) in coords_sp.items():
        if est in cam_min:
            cor = 'blue' if est not in [cam_min[0], cam_min[-1]] else ('green' if est == cam_min[0] else 'red')
            folium.CircleMarker([lat, lon], radius=7, color=cor, fill=True,
                                fill_color=cor, fill_opacity=0.9,
                                tooltip=est).add_to(m)
        else:
            folium.CircleMarker([lat, lon], radius=4, color='gray', fill=True,
                                fill_color='white', fill_opacity=0.7,
                                tooltip=est).add_to(m)

    # Legenda
    legend = """
    <div style="position:fixed;bottom:30px;left:30px;z-index:999;background:white;
                padding:10px 14px;border-radius:8px;box-shadow:0 2px 8px rgba(0,0,0,.3);font-size:13px">
      <b>Legenda — São Paulo</b><br>
      <span style="color:#1565C0">━━</span> Caminho mais curto<br>
      <span style="color:red">╌╌</span> Caminho mais longo<br>
      <span style="color:green">●</span> Origem &nbsp;
      <span style="color:red">●</span> Destino
    </div>"""
    m.get_root().html.add_child(folium.Element(legend))
    return m

m_sp = criar_mapa_sp(hora_vis=12)
m_sp.save('/mnt/user-data/outputs/mapa_saopaulo.html')
print("Mapa São Paulo salvo.")
m_sp


Mapa São Paulo salvo.


In [8]:
coords_beijing = {
    "Sihui East":         (39.9042, 116.5489),
    "Sihui":              (39.9042, 116.5351),
    "Guomao":             (39.9084, 116.4600),
    "Dawanglu":           (39.9084, 116.4750),
    "Yong'anli":          (39.9084, 116.4450),
    "Jianguomen":         (39.9084, 116.4300),
    "Beijing Station":    (39.9043, 116.4205),
    "Wangfujing":         (39.9121, 116.4070),
    "Tian'anmen East":    (39.9087, 116.3980),
    "Tian'anmen West":    (39.9087, 116.3900),
    "Xidan":              (39.9121, 116.3660),
    "Fuxingmen":          (39.9121, 116.3570),
    "Muxidi":             (39.9121, 116.3415),
    "Junshi Bowuguan":    (39.9121, 116.3315),
    "Gongzhufen":         (39.9100, 116.3170),
    "Wanshoulu":          (39.9100, 116.3050),
    "Jianguomen L2":      (39.9143, 116.4310),
    "Beijing Station L2": (39.9043, 116.4180),
    "Chongwenmen":        (39.8990, 116.4170),
    "Qianmen":            (39.8987, 116.3980),
    "Hepingmen":          (39.8987, 116.3860),
    "Xuanwumen":          (39.8990, 116.3750),
    "Changchunjie":       (39.8990, 116.3640),
    "Fuxingmen L2":       (39.9143, 116.3580),
    "Chegongzhuang":      (39.9267, 116.3405),
    "Chaoyangmen":        (39.9201, 116.4390),
    "Dongsishitiao":      (39.9301, 116.4335),
    "Dongzhimen":         (39.9349, 116.4379),
    "Guloudajie":         (39.9435, 116.4007),
    "Jishuitan":          (39.9455, 116.3739),
    "Xizhimen":           (39.9439, 116.3492),
    "Xidan L4":           (39.9135, 116.3665),
    "Xizhimen L4":        (39.9445, 116.3500),
    "Gongzhufen L10":     (39.9095, 116.3172),
    "Sanyuanqiao":        (39.9521, 116.4571),
}

def criar_mapa_beijing(hora_vis=12):
    res_vis = resolver_cidade(grafo_beijing, "Sihui East", "Xizhimen", hora=hora_vis)
    cam_min = res_vis['caminho_min']
    cam_max = res_vis['caminho_max']

    m = folium.Map(location=[39.920, 116.400], zoom_start=12, tiles='CartoDB positron')

    visitadas = set()
    for est, vizs in grafo_beijing.items():
        for viz, _ in vizs:
            par = tuple(sorted([est, viz]))
            if par not in visitadas:
                if est in coords_beijing and viz in coords_beijing:
                    folium.PolyLine([coords_beijing[est], coords_beijing[viz]],
                                    color='gray', weight=2, opacity=0.4).add_to(m)
                visitadas.add(par)

    coords_max = [coords_beijing[e] for e in cam_max if e in coords_beijing]
    if len(coords_max) >= 2:
        folium.PolyLine(coords_max, color='red', weight=3, opacity=0.7,
                        dash_array='8 4', tooltip='Caminho mais longo').add_to(m)

    coords_min = [coords_beijing[e] for e in cam_min if e in coords_beijing]
    if len(coords_min) >= 2:
        folium.PolyLine(coords_min, color='#1565C0', weight=5, opacity=0.9,
                        tooltip='Caminho mais curto').add_to(m)

    for est, (lat, lon) in coords_beijing.items():
        cor = 'gray'
        if est in cam_min:
            cor = 'green' if est == cam_min[0] else ('red' if est == cam_min[-1] else 'blue')
        folium.CircleMarker([lat, lon], radius=5 if cor=='gray' else 7,
                            color=cor, fill=True, fill_color=cor,
                            fill_opacity=0.85, tooltip=est).add_to(m)

    legend = """<div style="position:fixed;bottom:30px;left:30px;z-index:999;background:white;
                padding:10px 14px;border-radius:8px;box-shadow:0 2px 8px rgba(0,0,0,.3);font-size:13px">
      <b>Legenda — Beijing</b><br>
      <span style="color:#1565C0">━━</span> Caminho mais curto<br>
      <span style="color:red">╌╌</span> Caminho mais longo<br>
      <span style="color:green">●</span> Origem &nbsp;<span style="color:red">●</span> Destino
    </div>"""
    m.get_root().html.add_child(folium.Element(legend))
    return m

m_bj = criar_mapa_beijing(hora_vis=12)
m_bj.save('/mnt/user-data/outputs/mapa_beijing.html')
print("Mapa Beijing salvo.")
m_bj


Mapa Beijing salvo.


In [9]:
coords_sf = {
    "Dublin/Pleasanton":   (37.7018, -121.9002),
    "West Dublin":         (37.6996, -121.9284),
    "Castro Valley":       (37.6926, -122.0759),
    "Bay Fair":            (37.6971, -122.1264),
    "San Leandro":         (37.7022, -122.1611),
    "Fruitvale":           (37.7749, -122.2244),
    "Coliseum":            (37.7540, -122.1979),
    "Lake Merritt":        (37.7976, -122.2654),
    "West Oakland":        (37.8048, -122.2948),
    "Embarcadero":         (37.7929, -122.3969),
    "Montgomery St":       (37.7894, -122.4013),
    "Powell St":           (37.7843, -122.4079),
    "Civic Center":        (37.7797, -122.4148),
    "16th St Mission":     (37.7651, -122.4196),
    "24th St Mission":     (37.7522, -122.4182),
    "Glen Park":           (37.7329, -122.4340),
    "Balboa Park":         (37.7222, -122.4477),
    "Daly City":           (37.7062, -122.4690),
    "Embarcadero Y":       (37.7930, -122.3960),
    "12th St Oakland":     (37.8034, -122.2714),
    "19th St Oakland":     (37.8082, -122.2683),
    "MacArthur":           (37.8285, -122.2838),
    "Rockridge":           (37.8441, -122.2509),
    "Orinda":              (37.8784, -122.1837),
    "Lafayette":           (37.8937, -122.1241),
    "Walnut Creek":        (37.9055, -122.0678),
    "MacArthur G":         (37.8287, -122.2840),
    "South Hayward":       (37.6343, -122.0572),
    "Union City":          (37.5912, -122.0171),
    "Fremont":             (37.5573, -121.9763),
    "Balboa Y":            (37.7224, -122.4480),
    "Colma":               (37.6843, -122.4662),
}

def criar_mapa_sf(hora_vis=12):
    res_vis = resolver_cidade(grafo_sf, "Dublin/Pleasanton", "Daly City", hora=hora_vis)
    cam_min = res_vis['caminho_min']
    cam_max = res_vis['caminho_max']

    m = folium.Map(location=[37.77, -122.25], zoom_start=11, tiles='CartoDB positron')

    visitadas = set()
    for est, vizs in grafo_sf.items():
        for viz, _ in vizs:
            par = tuple(sorted([est, viz]))
            if par not in visitadas:
                if est in coords_sf and viz in coords_sf:
                    folium.PolyLine([coords_sf[est], coords_sf[viz]],
                                    color='gray', weight=2, opacity=0.4).add_to(m)
                visitadas.add(par)

    coords_max = [coords_sf[e] for e in cam_max if e in coords_sf]
    if len(coords_max) >= 2:
        folium.PolyLine(coords_max, color='red', weight=3, opacity=0.7,
                        dash_array='8 4', tooltip='Caminho mais longo').add_to(m)

    coords_min = [coords_sf[e] for e in cam_min if e in coords_sf]
    if len(coords_min) >= 2:
        folium.PolyLine(coords_min, color='#1565C0', weight=5, opacity=0.9,
                        tooltip='Caminho mais curto').add_to(m)

    for est, (lat, lon) in coords_sf.items():
        cor = 'gray'
        if est in cam_min:
            cor = 'green' if est == cam_min[0] else ('red' if est == cam_min[-1] else 'blue')
        folium.CircleMarker([lat, lon], radius=5 if cor=='gray' else 7,
                            color=cor, fill=True, fill_color=cor,
                            fill_opacity=0.85, tooltip=est).add_to(m)

    legend = """<div style="position:fixed;bottom:30px;left:30px;z-index:999;background:white;
                padding:10px 14px;border-radius:8px;box-shadow:0 2px 8px rgba(0,0,0,.3);font-size:13px">
      <b>Legenda — San Francisco BART</b><br>
      <span style="color:#1565C0">━━</span> Caminho mais curto<br>
      <span style="color:red">╌╌</span> Caminho mais longo<br>
      <span style="color:green">●</span> Origem &nbsp;<span style="color:red">●</span> Destino
    </div>"""
    m.get_root().html.add_child(folium.Element(legend))
    return m

m_sf = criar_mapa_sf(hora_vis=12)
m_sf.save('/mnt/user-data/outputs/mapa_sanfrancisco.html')
print("Mapa San Francisco salvo.")
m_sf


Mapa San Francisco salvo.


## [8] Interface Interativa — Input de Horário

Execute a célula abaixo para escolher a cidade e o horário de partida.


In [10]:
def rodar_interativo():
    print("=== Sistema de Rotas de Metrô — DP Checkpoint 2 ===\n")
    print("Cidades disponíveis:")
    print("  1. Beijing        (Sihui East → Xizhimen)")
    print("  2. San Francisco  (Dublin/Pleasanton → Daly City)")
    print("  3. São Paulo      (Tucuruvi → Capão Redondo)")
    escolha = input("\nEscolha a cidade (1/2/3): ").strip()
    hora = int(input("Informe o horário de partida (0-23): ").strip())

    cidade_map = {
        "1": ("Beijing",       grafo_beijing, "Sihui East",       "Xizhimen"),
        "2": ("San Francisco", grafo_sf,      "Dublin/Pleasanton","Daly City"),
        "3": ("São Paulo",     grafo_sp,      "Tucuruvi",         "Capão Redondo"),
    }

    if escolha not in cidade_map:
        print("Opção inválida."); return

    nome, grafo, orig, dest = cidade_map[escolha]
    fator = fator_horario(hora)
    print(f"\n🏙  {nome} | Fator horário ({hora}h): ×{fator}")

    import time, tracemalloc
    tracemalloc.start()
    t0 = time.perf_counter()
    res = resolver_cidade(grafo, orig, dest, hora)
    t1 = time.perf_counter()
    _, mem_pico = tracemalloc.get_traced_memory()
    tracemalloc.stop()

    print(f"\n✅ Caminho MAIS CURTO — custo: {res['custo_min']:.1f} min")
    print(f"   {' → '.join(res['caminho_min'])}")
    print(f"\n🔴 Caminho MAIS LONGO — custo: {res['custo_max']:.1f} min")
    print(f"   {' → '.join(res['caminho_max'])}")
    print(f"\n⏱  Tempo total: {(t1-t0)*1000:.2f} ms | Memória pico: {mem_pico/1024:.1f} KB")

# Descomente para rodar interativamente:
# rodar_interativo()

# Demonstração automática com horário 8h (pico manhã)
for nome_c, grafo_c, orig_c, dest_c in cidades:
    h = 8
    res = resolver_cidade(grafo_c, orig_c, dest_c, hora=h)
    print(f"\n🏙  {nome_c} @ {h}h (fator {fator_horario(h)})")
    print(f"  Mais curto: {res['custo_min']:.1f} min → {' → '.join(res['caminho_min'][:4])} ...")
    print(f"  Mais longo: {res['custo_max']:.1f} min ({len(res['caminho_max'])} estações)")



🏙  Beijing @ 8h (fator 1.5)
  Mais curto: 52.5 min → Sihui East → Sihui → Guomao → Dawanglu ...
  Mais longo: 114.0 min (26 estações)

🏙  San Francisco @ 8h (fator 1.5)
  Mais curto: 96.0 min → Dublin/Pleasanton → West Dublin → Castro Valley → Bay Fair ...
  Mais longo: 103.5 min (18 estações)

🏙  São Paulo @ 8h (fator 1.5)
  Mais curto: 63.0 min → Tucuruvi → Parada Inglesa → Jardim São Paulo → Carandiru ...
  Mais longo: 63.0 min (16 estações)


## [9] Análise de Complexidade e Conclusões

### Complexidade dos Algoritmos

| Algoritmo | Tempo (sem memo) | Tempo (com memo) | Espaço |
|---|---|---|---|
| Caminho mais curto | O(V!) | O(V · 2^V) | O(V · 2^V) |
| Caminho mais longo | O(V!) | NP-difícil (backtracking) | O(V) pilha |

### Conclusões sobre Memoização

1. **Ganho de desempenho:** A memoização reduz drasticamente o número de chamadas recursivas ao armazenar subproblemas já computados. Para grafos com ciclos e múltiplos caminhos, o speedup pode ser de ordens de magnitude.

2. **Trade-off memória × tempo:** A memoização consome mais memória (armazena o cache), mas reduz o tempo de CPU. Para grafos grandes (centenas de nós), o consumo de memória pode ser proibitivo.

3. **Caminho mais longo:** O problema do caminho mais longo simples não possui subestrutura ótima independente do conjunto de visitados — logo, a memoização clássica não se aplica diretamente. O backtracking é necessário, tornando-o exponencial.

4. **Fatores de horário:** O pico da tarde (×2,0) pode dobrar o custo total de uma viagem, incentivando viagens no horário de bônus (×0,6). Isso modela bem o comportamento real de penalidades de congestionamento.

5. **Grafo não-dirigido:** A escolha é adequada para redes de metrô convencionais. Redes com linhas expressas ou de mão única requereriam dígrafos.
